# BirdCLEF+ 2026 — Submission Notebook

**How to use:**
1. Train locally: `python train.py --all-folds`
2. Upload `outputs/fold*_best.pth` files to a private Kaggle Dataset named `birdclef2026-checkpoints`
3. Add that dataset to this notebook (+ the competition data)
4. Submit — Kaggle runs this notebook on the hidden test set

This notebook is **self-contained**: all model/transform code is inlined.

In [ ]:
# Install extra packages only if they are missing.
# Kaggle submission reruns may have internet disabled, so avoid unconditional pip.
import importlib.util
import subprocess
import sys

missing = [p for p in ['timm', 'librosa', 'soundfile'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)

In [ ]:
import ast
import glob
import math
import os
import random
from collections import OrderedDict
from typing import Dict, List, Optional, Tuple

import librosa
import numpy as np
import pandas as pd
import soundfile as sf
import timm
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm.notebook import tqdm

print('torch:', torch.__version__)
print('timm:', timm.__version__)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(42)

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────

DATA_DIR      = '/kaggle/input/birdclef-2026'
CKPT_DIR      = '/kaggle/input/birdclef2026-checkpoints'  # your uploaded weights
OUTPUT_DIR    = '/kaggle/working'

TAXONOMY_CSV          = f'{DATA_DIR}/taxonomy.csv'
SAMPLE_SUBMISSION_CSV = f'{DATA_DIR}/sample_submission.csv'
TEST_SOUNDSCAPES_DIR  = f'{DATA_DIR}/test_soundscapes'

# Audio
SAMPLE_RATE   = 32_000
CLIP_DURATION = 5.0
N_SAMPLES     = int(SAMPLE_RATE * CLIP_DURATION)

# Mel spectrogram
N_MELS      = 128
N_FFT       = 1024
HOP_LENGTH  = 320
FMIN        = 20.0
FMAX        = 16_000.0
TOP_DB      = 80.0

# Model
MODEL_NAME  = 'tf_efficientnet_b4_ns'
NUM_CLASSES = 234
IN_CHANNELS = 3
DROP_RATE   = 0.3
IMG_SIZE    = 256
PRETRAINED  = False   # load weights from checkpoint, not ImageNet

# Inference
INFER_BATCH_SIZE = 32
NUM_WORKERS      = 2

DEVICE = (
    torch.device('cuda') if torch.cuda.is_available()
    else torch.device('mps') if torch.backends.mps.is_available()
    else torch.device('cpu')
)
print('Device:', DEVICE)

In [ ]:
# ── Species info ──────────────────────────────────────────────────────────────

def build_species_info(taxonomy_csv_path):
    tax = pd.read_csv(taxonomy_csv_path)
    species_list = [str(x) for x in tax['primary_label'].tolist()]
    label_to_idx = {sp: i for i, sp in enumerate(species_list)}
    return {'species_list': species_list, 'label_to_idx': label_to_idx}

species_info = build_species_info(TAXONOMY_CSV)
SPECIES_LIST = species_info['species_list']
print(f'Species: {len(SPECIES_LIST)}')

In [ ]:
# ── Transforms ────────────────────────────────────────────────────────────────

_IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(3, 1, 1)
_IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(3, 1, 1)


def audio_transform_val(waveform):
    """Center-crop / tile-pad to N_SAMPLES, amplitude-normalise."""
    waveform = np.squeeze(waveform).astype(np.float32)
    if waveform.ndim != 1 or len(waveform) == 0:
        return np.zeros(N_SAMPLES, dtype=np.float32)
    if len(waveform) < N_SAMPLES:
        repeats = (N_SAMPLES // len(waveform)) + 1
        waveform = np.tile(waveform, repeats)
    start = (len(waveform) - N_SAMPLES) // 2
    waveform = waveform[start : start + N_SAMPLES]
    waveform /= (np.max(np.abs(waveform)) + 1e-6)
    return waveform


def mel_spec_transform(waveform):
    """waveform (N_SAMPLES,) → float32 (3, IMG_SIZE, IMG_SIZE) ImageNet-normalised."""
    mel = librosa.feature.melspectrogram(
        y=waveform, sr=SAMPLE_RATE, n_mels=N_MELS, n_fft=N_FFT,
        hop_length=HOP_LENGTH, fmin=FMIN, fmax=FMAX, power=2.0,
    )
    mel_db = librosa.power_to_db(mel, ref=1.0, top_db=TOP_DB)
    mel_norm = ((mel_db + TOP_DB) / TOP_DB).clip(0.0, 1.0).astype(np.float32)
    img = np.array(
        Image.fromarray((mel_norm * 255).astype(np.uint8)).resize(
            (IMG_SIZE, IMG_SIZE), Image.BILINEAR
        ),
        dtype=np.float32,
    ) / 255.0
    img = np.stack([img, img, img], axis=0)
    img = (img - _IMAGENET_MEAN) / _IMAGENET_STD
    return img

In [ ]:
# ── Inference Dataset ─────────────────────────────────────────────────────────
# Each soundscape is loaded ONCE from disk (on first access) and cached.
# Windows are sliced from the in-memory waveform — no repeated file I/O.
# num_workers=0 in the DataLoader keeps the cache in the main process.

class SoundscapeInferenceDataset(Dataset):
    def __init__(self, soundscapes_dir):
        self._cached_fpath = None
        self._cached_wav   = None
        self.items = []   # (fpath, start_sample, end_sample, row_id)
        ogg_files = sorted(f for f in os.listdir(soundscapes_dir) if f.lower().endswith('.ogg'))
        for fname in ogg_files:
            fpath = os.path.join(soundscapes_dir, fname)
            stem  = os.path.splitext(fname)[0]
            try:
                info = sf.info(fpath)
                duration = info.frames / info.samplerate
            except Exception:
                continue
            end = CLIP_DURATION
            while end <= duration + 1e-3:
                start_sample = int((end - CLIP_DURATION) * SAMPLE_RATE)
                end_sample   = int(end * SAMPLE_RATE)
                self.items.append((fpath, start_sample, end_sample, f'{stem}_{int(end)}'))
                end += CLIP_DURATION

    def _load_file(self, fpath):
        if fpath != self._cached_fpath:
            self._cached_wav, _ = librosa.load(fpath, sr=SAMPLE_RATE, mono=True)
            self._cached_fpath  = fpath
        return self._cached_wav

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        fpath, start_sample, end_sample, row_id = self.items[idx]
        try:
            wav    = self._load_file(fpath)
            window = wav[start_sample:end_sample].copy()
            if len(window) == 0:
                raise ValueError('empty')
        except Exception:
            window = np.zeros(N_SAMPLES, dtype=np.float32)
        window = audio_transform_val(window)
        img    = mel_spec_transform(window)
        return torch.from_numpy(img), row_id

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────────

class BirdModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            MODEL_NAME, pretrained=PRETRAINED, in_chans=IN_CHANNELS,
            num_classes=0, global_pool='avg', drop_rate=DROP_RATE,
        )
        self.classifier = nn.Linear(self.backbone.num_features, NUM_CLASSES)

    def forward(self, x):
        return self.classifier(self.backbone(x))


def load_model(ckpt_path):
    model = BirdModel()
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model.to(DEVICE)
    model.eval()
    snd_cmap = ckpt.get('snd_cmap', ckpt.get('val_cmap', '?'))
    clip_cmap = ckpt.get('clip_cmap', '?')
    print(f'  {os.path.basename(ckpt_path)}  snd_cmAP={snd_cmap}  clip_cmAP={clip_cmap}')
    return model

In [ ]:
# ── Load checkpoints ──────────────────────────────────────────────────────────

ckpt_paths = sorted(glob.glob(os.path.join(CKPT_DIR, 'fold*_best.pth')))
if not ckpt_paths:
    raise FileNotFoundError(
        f'No checkpoints found in {CKPT_DIR}. '
        'Upload your trained weights as a Kaggle Dataset named "birdclef2026-checkpoints".'
    )

print(f'Loading {len(ckpt_paths)} checkpoint(s):')
models = [load_model(p) for p in ckpt_paths]

In [ ]:
# ── Inference ─────────────────────────────────────────────────────────────────

dataset = SoundscapeInferenceDataset(TEST_SOUNDSCAPES_DIR)
loader  = DataLoader(
    dataset,
    batch_size=INFER_BATCH_SIZE,
    shuffle=False,
    num_workers=0,        # must be 0: file cache lives in main process
    pin_memory=(DEVICE.type == 'cuda'),
)
print(f'Total 5-second windows: {len(dataset)}')

all_row_ids = []
all_probs   = []

with torch.no_grad():
    for images, row_ids in tqdm(loader, desc='Inference'):
        images = images.to(DEVICE)
        batch_probs = []
        for m in models:
            logits = m(images)
            batch_probs.append(torch.sigmoid(logits).cpu().float().numpy())
        avg_probs = np.mean(batch_probs, axis=0)
        all_row_ids.extend(row_ids)
        all_probs.append(avg_probs)

if not all_probs:
    raise RuntimeError('No windows processed — test_soundscapes/ appears empty.')

all_probs = np.concatenate(all_probs, axis=0)
print(f'Predictions shape: {all_probs.shape}')

In [ ]:
# ── Build and save submission ─────────────────────────────────────────────────

sample = pd.read_csv(SAMPLE_SUBMISSION_CSV)
if len(all_row_ids) == 0:
    # Local public copy has an empty test_soundscapes directory.
    # On the hidden rerun, Kaggle populates it and this branch is skipped.
    sub = sample.copy()
else:
    sub = pd.DataFrame(all_probs, columns=SPECIES_LIST)
    sub.insert(0, 'row_id', all_row_ids)
    sub = sub.reindex(columns=sample.columns, fill_value=0.0)

out_path = os.path.join(OUTPUT_DIR, 'submission.csv')
sub.to_csv(out_path, index=False)
print(f'Saved → {out_path}')
print(f'Shape: {sub.shape}')
sub.head()